<div align="center">

<a href="https://colab.research.google.com/github/utkukose/machine-ethics-ai-safety-NB-lecture/blob/main/weeks/week-06/NB06_fairness_mitigation.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 06: Algorithmic Fairness II: Mitigation and Representational Harms

**Machine Ethics and Artificial Intelligence Safety (11118BLG001)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/machine-ethics-ai-safety-NB-lecture/weeks/week-06/lab.html>.

## Overview

Measuring unfairness is the first step. This week covers interventions before, during and after training, the limits of removing protected attributes, counterfactual reasoning about fairness and the representational harms that language technologies can cause [1, 2, 3, 4].

## Learning outcomes

By the end of the week, students are expected to apply reweighing and group-specific thresholds, to explain the reductions approach to fair classification, to show why fairness through unawareness fails in the presence of proxies, to describe counterfactual fairness, and to analyse gender bias in word embeddings and the limits of debiasing.

## Three points of intervention

Mitigation methods act on the data, on the learning algorithm or on the predictions. Kamiran and Calders proposed pre-processing techniques, including reweighing: Each training example receives the weight P(group) P(label) / P(group, label), which makes group and label statistically independent in the weighted data [1]. Pre-processing is simple and model-agnostic, but it cannot guarantee a particular fairness criterion at test time.

In-processing changes the objective. Agarwal and colleagues reduced fair classification under constraints such as demographic parity or equalized odds to a sequence of cost-sensitive classification problems, whose solutions are combined into a randomised classifier with provable guarantees [2]. Post-processing changes only the decision rule. Hardt, Price and Srebro showed how to derive group-specific, possibly randomised thresholds that achieve equalized odds or equal opportunity from any score, without retraining [3]. The open-source Fairlearn toolkit implements assessment tools and several of these mitigation methods [5]. Figure 6.1 compares the three routes on the lending data of Week 5.

![Figure 6.1](https://raw.githubusercontent.com/utkukose/machine-ethics-ai-safety-NB-lecture/main/weeks/week-06/figures/w06_fig1.png)

*Figure 6.1. Accuracy against the equal opportunity gap on the synthetic lending data of Week 5 for a threshold sweep without mitigation, with reweighing and with group-specific thresholds [1, 3].*

## Proxies and counterfactual fairness

A tempting remedy is to delete the protected attribute. This fairness through unawareness usually fails, because other features act as proxies: Postal codes, schools, names and purchase histories can encode group membership. The Week 3 notebook showed the same effect for a morally irrelevant attribute, and the Week 5 notebook showed unequal error rates for a model that never saw the group.

Kusner and colleagues defined counterfactual fairness with a causal model: A prediction is counterfactually fair if it would stay the same in a counterfactual world where the individual's protected attribute had been different, with everything not caused by the attribute held fixed [6]. The definition captures the intuition behind many discrimination complaints. Its price is a causal model of how the protected attribute influences the other features, and conclusions depend on assumptions that must be defended.

## Representational harms in language technology

Fairness is not only about decisions. Bolukbasi and colleagues showed that word embeddings trained on news text completed analogies such as man is to computer programmer as woman is to homemaker, and they proposed hard debiasing: Identify a gender direction and remove it from words that should be gender-neutral [4]. Caliskan, Bryson and Narayanan adapted the implicit association test to embeddings and reproduced a range of documented human biases in their word embedding association test [7].

Debiasing turned out to be shallow. Gonen and Goldberg showed that after projection, words that had been stereotypically gendered still clustered together and that the removed bias could be recovered by a classifier from the remaining dimensions [8]. Bender and colleagues warned that ever larger language models absorb the viewpoints over-represented in their training data and called for careful curation and documentation [9]. The lab reproduces both the analogy effect and the residual bias in a two-dimensional toy space.

## Choosing and documenting a mitigation

No mitigation is free. Each method moves the operating point of a system, and the right point depends on the costs of errors for each group and on legal constraints, such as rules on using protected attributes at decision time. A defensible process names the criterion, explains why it fits the decision, reports the accuracy and fairness trade-off with uncertainty and records the choice in documentation such as a model card, which Week 14 covers [10].

> **Pause and reflect.** Group-specific thresholds equalise true positive rates but treat identical scores differently. Is that fair? Consider the argument from the perspective of an applicant in each group.

# Hands-on lab

The notebook applies reweighing and group-specific thresholds to the lending data of Week 5, compares their fairness and accuracy, and shows how to run the same assessment with Fairlearn when it is installed [1, 3, 5].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. The lending data of Week 5

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(SEED)
n = 6000
group = rng.integers(0, 2, n)
income = rng.normal(50 - 8 * group, 12, n)
debt = rng.normal(20, 8, n)
history = rng.normal(0.6 - 0.1 * group, 0.2, n)
logit = 0.08 * (income - 45) - 0.09 * (debt - 20) + 3.0 * (history - 0.55)
repaid = (rng.uniform(size=n) < 1 / (1 + np.exp(-logit))).astype(int)
X = np.column_stack([income, debt, history])
X_tr, X_te, y_tr, y_te, g_tr, g_te = train_test_split(X, repaid, group, test_size=0.4, random_state=0)

def report(pred, name):
    tpr = [pred[(g_te == k) & (y_te == 1)].mean() for k in (0, 1)]
    sel = [pred[g_te == k].mean() for k in (0, 1)]
    return {"method": name, "accuracy": np.mean(pred == y_te), "EO gap": tpr[0] - tpr[1], "DP gap": sel[0] - sel[1]}

base = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
s_base = base.predict_proba(X_te)[:, 1]
results = [report((s_base >= 0.5).astype(int), "no mitigation")]
pd.DataFrame(results).round(3)

## 2. Pre-processing by reweighing

### Your turn, exercise 1

Compute the reweighing weight for each (group, label) cell, w(g, y) = P(g) P(y) / P(g, y), from the training data. Return a 2 x 2 array indexed as `W[g, y]`.

In [ ]:
W = None  # your 2 x 2 array

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _reweigh_reference():
    Wr = np.zeros((2, 2))
    for a in (0, 1):
        for b in (0, 1):
            Wr[a, b] = np.mean(g_tr == a) * np.mean(y_tr == b) / np.mean((g_tr == a) & (y_tr == b))
    return Wr

In [ ]:
check("Exercise 1", W, _reweigh_reference())
Wref = _reweigh_reference()
sample_w = Wref[g_tr, y_tr]
rew = LogisticRegression(max_iter=1000).fit(X_tr, y_tr, sample_weight=sample_w)
s_rew = rew.predict_proba(X_te)[:, 1]
results.append(report((s_rew >= 0.5).astype(int), "reweighing"))
pd.DataFrame(results).round(3)

## 3. Post-processing with group-specific thresholds

### Your turn, exercise 2

Write a function that returns the threshold for group B whose true positive rate is closest to that of group A at threshold 0.5. Search over `np.linspace(0.05, 0.95, 181)`.

In [ ]:
def matching_threshold(scores, t_a=0.5):
    # Your code here
    return 0.5

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _matching_reference(scores, t_a=0.5):
    tpr_a = np.mean(scores[(g_te == 0) & (y_te == 1)] >= t_a)
    cands = np.linspace(0.05, 0.95, 181)
    return cands[np.argmin([abs(np.mean(scores[(g_te == 1) & (y_te == 1)] >= c) - tpr_a) for c in cands])]

In [ ]:
check("Exercise 2", matching_threshold(s_base), _matching_reference(s_base))
t_b = _matching_reference(s_base)
post = np.where(g_te == 0, s_base >= 0.5, s_base >= t_b).astype(int)
results.append(report(post, f"group thresholds (B: {t_b:.3f})"))
pd.DataFrame(results).round(3)

## 4. The same assessment with Fairlearn

Fairlearn is not preinstalled in Colab. Run `!pip install fairlearn` first to use this cell.

In [ ]:
try:
    from fairlearn.metrics import MetricFrame, selection_rate, true_positive_rate
    mf = MetricFrame(metrics={"selection rate": selection_rate, "TPR": true_positive_rate},
                     y_true=y_te, y_pred=post, sensitive_features=np.where(g_te == 0, "A", "B"))
    display(mf.by_group)
    print("differences:", mf.difference().round(3).to_dict())
except ImportError:
    print("Fairlearn is not installed. Run: pip install fairlearn")

Group-specific thresholds close the equal opportunity gap almost completely. Reweighing changes the model less and closes the gap only partly. Write down which of the three models you would deploy and why.

## Visual exploration: How strong must a proxy be?

The model below never sees the protected attribute. It sees a skill score and a proxy whose correlation with the attribute varies from 0 to 0.9, while historical labels favour one group. Change the strength of the historical bias and observe how the selection-rate gap of the group-blind model responds.

In [ ]:
from sklearn.linear_model import LogisticRegression
rng = np.random.default_rng(SEED)
rhos, gaps = np.linspace(0, 0.9, 10), []
for rho in rhos:
    n = 6000
    A = rng.binomial(1, 0.4, n)
    proxy = rho * (A - 0.4) / np.sqrt(0.24) + np.sqrt(1 - rho ** 2) * rng.normal(0, 1, n)
    skill = rng.normal(0, 1, n)
    y = (skill + 0.8 * A + rng.normal(0, 0.5, n) > 0.5).astype(int)   # historical labels favour group 1
    pred = LogisticRegression().fit(np.c_[skill, proxy], y).predict(np.c_[skill, proxy])
    gaps.append(pred[A == 1].mean() - pred[A == 0].mean())
plt.plot(rhos, gaps, "o-")
plt.xlabel("correlation of the proxy with the protected attribute"); plt.ylabel("selection-rate gap of a group-blind model")
plt.title("Removing the attribute does not remove its influence"); plt.show()

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/machine-ethics-ai-safety-NB-lecture/weeks/week-06/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Which of the three mitigation routes would be legally and practically feasible in your country for credit decisions, and why?
2. In the lab, the analogy test changed before the residual bias disappeared. What does that suggest about using a single bias metric?
3. Describe a representational harm that a language model could cause in Turkish or another language you use, and how you would test for it.

## Weekly task and submission

Apply one mitigation method from this week to a model of your choice (the notebook model is acceptable). In about 400 words, report fairness and accuracy before and after, discuss who gains and who loses, and state which method you would deploy. Attach the executed notebook.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**Representational harms in Turkish language technologies.** Review the evidence on gender bias in machine translation and language models for Turkish, where the pronoun o carries no gender, and evaluate proposed mitigations [4, 7].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] Kamiran, F., & Calders, T. (2012). Data preprocessing techniques for classification without discrimination. *Knowledge and Information Systems*, 33(1), 1-33. <https://doi.org/10.1007/s10115-011-0463-8>

[2] Agarwal, A., Beygelzimer, A., Dudík, M., Langford, J., & Wallach, H. (2018). A reductions approach to fair classification. In *Proceedings of the 35th International Conference on Machine Learning, PMLR 80* (pp. 60-69).

[3] Hardt, M., Price, E., & Srebro, N. (2016). Equality of opportunity in supervised learning. In *Advances in Neural Information Processing Systems 29*.

[4] Bolukbasi, T., Chang, K.-W., Zou, J. Y., Saligrama, V., & Kalai, A. T. (2016). Man is to computer programmer as woman is to homemaker? Debiasing word embeddings. In *Advances in Neural Information Processing Systems 29*.

[5] Weerts, H., Dudík, M., Edgar, R., Jalali, A., Lutz, R., & Madaio, M. (2023). Fairlearn: Assessing and improving fairness of AI systems. *Journal of Machine Learning Research*, 24(257), 1-8.

[6] Kusner, M. J., Loftus, J., Russell, C., & Silva, R. (2017). Counterfactual fairness. In *Advances in Neural Information Processing Systems 30*.

[7] Caliskan, A., Bryson, J. J., & Narayanan, A. (2017). Semantics derived automatically from language corpora contain human-like biases. *Science*, 356(6334), 183-186. <https://doi.org/10.1126/science.aal4230>

[8] Gonen, H., & Goldberg, Y. (2019). Lipstick on a pig: Debiasing methods cover up systematic gender biases in word embeddings but do not remove them. In *Proceedings of the 2019 Conference of the North American Chapter of the Association for Computational Linguistics (NAACL-HLT)* (pp. 609-614). Association for Computational Linguistics. <https://arxiv.org/abs/1903.03862>

[9] Bender, E. M., Gebru, T., McMillan-Major, A., & Shmitchell, S. (2021). On the dangers of stochastic parrots: Can language models be too big?. In *Proceedings of the 2021 ACM Conference on Fairness, Accountability, and Transparency (FAccT '21)* (pp. 610-623). <https://doi.org/10.1145/3442188.3445922>

[10] Mitchell, M., Wu, S., Zaldivar, A., Barnes, P., Vasserman, L., Hutchinson, B., Spitzer, E., Raji, I. D., & Gebru, T. (2019). Model cards for model reporting. In *Proceedings of the Conference on Fairness, Accountability, and Transparency (FAT* '19)* (pp. 220-229). ACM. <https://doi.org/10.1145/3287560.3287596>